In [3]:
# 1. Import libraries
import pandas as pd
from pandasql import sqldf

# 2. Define our SQL function
run_sql = lambda q: sqldf(q, globals())

# 3. Load the dataset
file_path = "/kaggle/input/netflix-shows/netflix_titles.csv"
Netflix = pd.read_csv(file_path)

# 4. Clean up column names with spaces for easier SQL querying
Netflix.rename(columns={
    'show_id': 'show_id', 'type': 'type', 'title': 'title',
    'director': 'director', 'cast': 'cast', 'country': 'country',
    'date_added': 'date_added', 'release_year': 'release_year',
    'rating': 'rating', 'duration': 'duration', 'listed_in': 'listed_in',
    'description': 'description'
}, inplace=True)

# 5. Drop rows with no date_added to ensure our date queries work
Netflix.dropna(subset=['date_added'], inplace=True)

print("--- Netflix Table is Loaded and Ready for Analysis ---")
print("--- First 5 Rows ---")
print(Netflix.head()) 

--- Netflix Table is Loaded and Ready for Analysis ---
--- First 5 Rows ---
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  Septemb

In [4]:
# 1: Top 10 Countries by Content Volume

from pandasql import sqldf
run_sql = lambda q: sqldf(q, globals())

query1 = """
SELECT
    country,
    COUNT(show_id) AS number_of_titles
FROM
    Netflix
WHERE
    country IS NOT NULL
GROUP BY
    country
ORDER BY
    number_of_titles DESC
LIMIT 10;
"""
results1 = run_sql(query1)
print(results1)

          country  number_of_titles
0   United States              2812
1           India               972
2  United Kingdom               418
3           Japan               244
4     South Korea               199
5          Canada               181
6           Spain               145
7          France               124
8          Mexico               110
9           Egypt               106


In [7]:
# 2: How Content Added Has Trended Over the Years

from pandasql import sqldf
run_sql = lambda q: sqldf(q, globals())

query2 = """
SELECT
    TRIM(SUBSTR(date_added, -4)) AS year_added,
    COUNT(show_id) AS number_of_titles
FROM
    Netflix
GROUP BY
    year_added
ORDER BY
    year_added ASC;
"""
results2 = run_sql(query2)
print(results2)


   year_added  number_of_titles
0        2008                 2
1        2009                 2
2        2010                 1
3        2011                13
4        2012                 3
5        2013                11
6        2014                24
7        2015                82
8        2016               429
9        2017              1188
10       2018              1649
11       2019              2016
12       2020              1879
13       2021              1498


In [8]:
# 3: The Most Common Genre for both Movies and TV Shows

from pandasql import sqldf
run_sql = lambda q: sqldf(q, globals())

query3 = """
WITH GenreRanking AS (
    SELECT
        type,
        listed_in AS genre,
        COUNT(*) as title_count,
        RANK() OVER (PARTITION BY type ORDER BY COUNT(*) DESC) as rank
    FROM
        Netflix
    GROUP BY
        type,
        genre
)
SELECT
    type,
    genre,
    title_count
FROM
    GenreRanking
WHERE
    rank = 1;
"""
results3 = run_sql(query3)
print(results3)

      type                         genre  title_count
0    Movie  Dramas, International Movies          362
1  TV Show                      Kids' TV          219


In [9]:
# 4: What is the average duration of Movies on Netflix?

from pandasql import sqldf
run_sql = lambda q: sqldf(q, globals())

query4 = """
SELECT
    AVG(CAST(REPLACE(duration, ' min', '') AS INTEGER)) AS average_movie_duration_minutes
FROM
    Netflix
WHERE
    type = 'Movie'
    AND duration LIKE '%min%'; -- Ensure we only process movie durations in minutes
"""
results4 = run_sql(query4)
print(results4)

   average_movie_duration_minutes
0                       99.577187
